# MapReduce-Reloaded: the demo, end to end

This notebook runs the demo in `docs/runbook.md` §5 against the Compose
cluster: upload the corpus, submit word count, read the output, plot it,
run a second program, look at the metrics, and check local mode against the
cluster.

**Before you start**, on the VM, from the repository root:

```sh
make cluster-up mr      # the cluster and the client
make upload-example     # the corpus and a 10 MB web log
```

The kernel needs `matplotlib` for the plot (`pip install matplotlib`).
Every cell is safe to rerun; each run uses a timestamped output directory.


## 1. Setup


In [ ]:
import subprocess
import time
import urllib.request
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "bin" / "mr").exists():
    ROOT = Path.home() / "MapReduce-Reloaded"
MR = ROOT / "bin" / "mr"
RUN = int(time.time())
COMPOSE = ["docker", "compose", "-p", "mapreduce", "-f", "deploy/compose/compose.yaml"]
METRICS = "http://127.0.0.1:18780/metrics"


def run(*args, check=True):
    """Runs a command in the repository root and returns the result."""
    result = subprocess.run([str(a) for a in args], cwd=ROOT, text=True, capture_output=True)
    if check and result.returncode != 0:
        raise RuntimeError(f"{' '.join(map(str, args))} failed:\n{result.stderr}")
    return result


def metrics_lines():
    """The coordinator's /metrics, as text lines."""
    with urllib.request.urlopen(METRICS, timeout=5) as response:
        return response.read().decode().splitlines()


def fields(text):
    """A status reply, as a dict: job-1\tSUCCEEDED, map\t4/4, counters."""
    return dict(line.split("\t", 1) for line in text.splitlines() if "\t" in line)


assert MR.exists(), "bin/mr is missing: run `make cluster-up mr` first"
print(f"repository: {ROOT}")


## 2. Check the cluster


In [ ]:
ps = run(*COMPOSE, "ps", "--format", "{{.Service}}\t{{.Status}}")
print(ps.stdout or "no containers: run `make cluster-up mr`")
print([line for line in metrics_lines() if line.startswith("mr_workers ")][0])


## 3. Upload the corpus


In [ ]:
run(MR, "upload", "testdata/corpus/*.txt", "/data/input/notebook-corpus")
print(sorted(p.name for p in (ROOT / "data/input/notebook-corpus").iterdir()))


## 4. Submit word count


In [ ]:
result = run(
    MR, "submit", "-m", "4", "-r", "2", "-wait", "wordcount",
    f"/data/out/notebook-wc-{RUN}", "/data/input/notebook-corpus/*.txt",
)
print(result.stdout)
status = fields(result.stdout)
print({key: status[key] for key in ("map", "reduce", "seconds") if key in status})
print({key: value for key, value in status.items() if key.endswith((".records", ".bytes"))})


## 5. Read the output


In [ ]:
output = run(MR, "cat", f"/data/out/notebook-wc-{RUN}").stdout
words = [line.split("\t") for line in output.splitlines()]
top = sorted(((word, int(count)) for word, count in words), key=lambda pair: pair[1], reverse=True)[:10]
for word, count in top:
    print(f"{word:>16} {count:>8,}")


## 6. Plot the ten most frequent words


In [ ]:
import matplotlib.pyplot as plt

figure, axis = plt.subplots(figsize=(7, 3.5))
axis.barh([word for word, _ in reversed(top)], [count for _, count in reversed(top)], color="#3b6ea5")
axis.set_xlabel("occurrences")
axis.set_title("Ten most frequent words, cluster word count")
figure.tight_layout()
plt.show()


## 7. A second program: the most requested pages


In [ ]:
assert (ROOT / "data/input/weblog/access.log").exists(), "run `make upload-example` first"
run(
    MR, "submit", "-m", "2", "-wait", "urlfreq",
    f"/data/out/notebook-urls-{RUN}", "/data/input/weblog/access.log",
)
urls = [line.split("\t") for line in run(MR, "cat", f"/data/out/notebook-urls-{RUN}").stdout.splitlines()]
for url, count in sorted(((url, int(n)) for url, n in urls), key=lambda pair: pair[1], reverse=True)[:5]:
    print(f"{count:>8,}  {url}")


## 8. The coordinator's metrics


In [ ]:
for line in metrics_lines():
    if line.startswith("mr_workers ") or 'state="completed"' in line:
        print(line)


## 9. Local mode is the oracle

Local mode runs the same job in one process, without Docker. Its output must
match the cluster's, byte for byte (README, "Run a job on one machine").


In [ ]:
local = f"data/out/notebook-local-{RUN}"
run(MR, "run", "-local", "-m", "4", "-r", "2", "wordcount", local, "testdata/corpus/*.txt")
local_output = run(MR, "cat", f"/{local}").stdout
print("local mode and the cluster agree:", sorted(local_output.splitlines()) == sorted(output.splitlines()))


## 10. Clean up


In [ ]:
for path in (
    f"data/out/notebook-wc-{RUN}",
    f"data/out/notebook-urls-{RUN}",
    f"data/out/notebook-local-{RUN}",
    "data/input/notebook-corpus",
):
    run("rm", "-rf", path)
print("removed this run's input and output")
